In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import torch.optim as optim

from torch.nn.utils.fusion import fuse_conv_bn_eval

from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import onnx

import matplotlib.pyplot as plt
import numpy as np

from datetime import datetime
from time import time
import copy

In [2]:
import json_to_pytorch

In [3]:
model = json_to_pytorch.json_to_pytorch("test.json")

In [4]:
model

Sequential(
  (0): Conv2d(3, 4, kernel_size=(3, 3), stride=(1, 1))
  (1): FrozenBatchNorm()
  (2): ReLU()
  (3): Conv2d(4, 2, kernel_size=(3, 3), stride=(1, 1))
  (4): GELU(approximate='none')
  (5): Flatten(start_dim=1, end_dim=-1)
  (6): Linear(in_features=32, out_features=16, bias=True)
  (7): ChebyshevPoly()
  (8): Linear(in_features=16, out_features=4, bias=True)
)

In [5]:
test_input = torch.ones((1,3,8,8))

In [6]:
test_input

tensor([[[[1., 1., 1., 1., 1., 1., 1., 1.],
          [1., 1., 1., 1., 1., 1., 1., 1.],
          [1., 1., 1., 1., 1., 1., 1., 1.],
          [1., 1., 1., 1., 1., 1., 1., 1.],
          [1., 1., 1., 1., 1., 1., 1., 1.],
          [1., 1., 1., 1., 1., 1., 1., 1.],
          [1., 1., 1., 1., 1., 1., 1., 1.],
          [1., 1., 1., 1., 1., 1., 1., 1.]],

         [[1., 1., 1., 1., 1., 1., 1., 1.],
          [1., 1., 1., 1., 1., 1., 1., 1.],
          [1., 1., 1., 1., 1., 1., 1., 1.],
          [1., 1., 1., 1., 1., 1., 1., 1.],
          [1., 1., 1., 1., 1., 1., 1., 1.],
          [1., 1., 1., 1., 1., 1., 1., 1.],
          [1., 1., 1., 1., 1., 1., 1., 1.],
          [1., 1., 1., 1., 1., 1., 1., 1.]],

         [[1., 1., 1., 1., 1., 1., 1., 1.],
          [1., 1., 1., 1., 1., 1., 1., 1.],
          [1., 1., 1., 1., 1., 1., 1., 1.],
          [1., 1., 1., 1., 1., 1., 1., 1.],
          [1., 1., 1., 1., 1., 1., 1., 1.],
          [1., 1., 1., 1., 1., 1., 1., 1.],
          [1., 1., 1., 1., 1

In [25]:
input3 = torch.tensor([[[[ 4.4436e-02,  1.2455e+00,  6.1157e-01,  1.0718e+00,  7.9047e-01,
                   2.3047e+00, -8.2357e-01,  9.3302e-02],
                 [-1.9668e+00,  8.1368e-01, -1.4142e+00, -9.5656e-02, -9.3935e-01,
                   7.0205e-01,  1.0099e+00, -2.3480e-01],
                 [ 3.0336e-01, -3.8012e-01, -8.3915e-01, -8.7246e-01,  1.3385e-01,
                  -1.0058e+00,  8.1397e-02,  1.5392e-01],
                 [ 1.0391e+00, -2.7163e-01, -2.9119e-02,  6.3678e-01,  2.4545e-02,
                   3.6523e-01, -3.6626e-01,  1.2044e-02],
                 [-2.8830e-01,  1.4077e-01, -3.9139e-01,  3.2097e-01, -3.9817e-01,
                   5.5241e-01, -1.5955e-01, -3.3260e-01],
                 [ 5.0310e-01, -4.0660e-01,  9.4994e-01, -4.1674e-01,  2.5218e-01,
                  -9.2530e-01,  1.2157e+00, -1.7149e+00],
                 [-8.9817e-01, -1.9178e-01, -8.7271e-01, -1.0396e+00, -1.1740e-01,
                  -1.3195e-01,  1.7033e-01,  6.7498e-01],
                 [ 7.9944e-01, -4.0011e-01, -1.2451e+00,  3.7919e-02,  2.5060e-01,
                  -3.7890e-01, -6.2014e-02, -6.6048e-01]],

                [[-1.0996e+00,  3.2182e+00,  9.5901e-01,  7.3111e-01,  1.8781e+00,
                  -3.1200e+00, -7.0371e-01, -9.6519e-01],
                 [ 2.1280e+00,  3.7126e-01, -1.1071e+00,  5.8979e-01, -6.7152e-01,
                   5.7031e-01,  3.2596e-01, -5.8595e-01],
                 [ 1.0356e+00, -4.1576e-01, -5.5026e-01,  3.9424e-01, -1.3170e-01,
                   5.6984e-01,  2.8612e-01,  9.5750e-02],
                 [ 1.7463e-01,  7.5853e-03, -1.1593e+00, -1.2891e+00, -2.8834e-02,
                  -3.3435e-01,  1.2481e+00,  1.8938e+00],
                 [-1.5252e-01, -6.5878e-01,  1.0264e-01, -6.7660e-01,  7.3020e-02,
                   9.9183e-01,  7.6018e-02, -1.6335e-01],
                 [ 9.8675e-01,  1.7565e+00, -6.5719e-01, -1.0794e+00, -1.5500e+00,
                   1.5881e+00, -1.7239e+00,  2.5569e-01],
                 [ 1.0791e+00,  8.7130e-01, -5.7949e-01,  6.7166e-01, -1.1020e+00,
                  -2.0487e+00,  1.5633e+00, -1.3794e+00],
                 [ 3.2025e-01, -2.2004e-01,  3.3018e-01,  1.9393e+00, -7.3435e-01,
                  -2.8490e-01,  6.0018e-01,  6.2037e-01]],

                [[-8.4940e-01, -1.2518e+00, -8.9065e-02,  9.4743e-01, -6.8050e-02,
                   1.0524e+00, -3.1379e-01,  8.6158e-01],
                 [ 1.2360e+00, -5.2726e-01, -6.8569e-01,  1.9485e-01,  1.2489e+00,
                  -7.0725e-01, -6.4146e-01,  2.7735e-01],
                 [-9.9147e-01,  5.4836e-01, -4.7740e-01, -1.4808e+00,  5.3619e-01,
                  -9.0408e-01,  1.2946e+00, -7.0238e-01],
                 [ 3.4375e-01, -3.2814e-01,  1.3189e+00, -2.9078e-03,  2.4637e-01,
                  -1.4270e+00, -1.8242e-02, -2.1706e+00],
                 [ 8.8646e-02, -6.4560e-01, -1.3097e+00, -3.5223e-01,  2.3314e-01,
                  -1.1131e+00, -1.2108e-01, -1.2666e-02],
                 [ 5.7811e-01, -1.6809e+00, -1.2356e+00,  1.3245e+00,  1.2095e+00,
                   1.1474e+00,  6.0248e-01,  1.1690e-02],
                 [ 2.4339e-01,  5.7339e-01, -2.2917e-01, -1.9299e-01, -1.4834e+00,
                  -1.9611e+00, -4.5528e-01,  1.7276e-01],
                 [ 2.2091e+00, -4.4662e-01,  4.1550e-01,  2.1087e-01,  2.8708e-01,
                  -1.4026e+00,  1.2779e+00, -8.0410e-01]]]])

In [7]:
input2 = torch.zeros((1,3,8,8))

In [8]:
module_list = list(model.children())
module_list

[Conv2d(3, 4, kernel_size=(3, 3), stride=(1, 1)),
 FrozenBatchNorm(),
 ReLU(),
 Conv2d(4, 2, kernel_size=(3, 3), stride=(1, 1)),
 GELU(approximate='none'),
 Flatten(start_dim=1, end_dim=-1),
 Linear(in_features=32, out_features=16, bias=True),
 ChebyshevPoly(),
 Linear(in_features=16, out_features=4, bias=True)]

In [27]:
model_part = nn.Sequential(*module_list[0:1])

In [28]:
model_part

Sequential(
  (0): Conv2d(3, 4, kernel_size=(3, 3), stride=(1, 1))
)

In [29]:
module_list[6]

Linear(in_features=32, out_features=16, bias=True)

In [30]:
model_part(test_input[0:1]).shape

torch.Size([1, 4, 6, 6])

In [33]:
input2[0,0,0,0] = 1
input2[0,0,0,1] = 1

In [34]:
input2

tensor([[[[1., 1., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0.]],

         [[0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0.]],

         [[0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0

In [35]:
model_part(input2[0:1])

tensor([[[[ 0.6930,  0.0614, -1.3773, -1.3773, -1.3773, -1.3773],
          [-1.3773, -1.3773, -1.3773, -1.3773, -1.3773, -1.3773],
          [-1.3773, -1.3773, -1.3773, -1.3773, -1.3773, -1.3773],
          [-1.3773, -1.3773, -1.3773, -1.3773, -1.3773, -1.3773],
          [-1.3773, -1.3773, -1.3773, -1.3773, -1.3773, -1.3773],
          [-1.3773, -1.3773, -1.3773, -1.3773, -1.3773, -1.3773]],

         [[-2.5286, -2.4881, -2.5929, -2.5929, -2.5929, -2.5929],
          [-2.5929, -2.5929, -2.5929, -2.5929, -2.5929, -2.5929],
          [-2.5929, -2.5929, -2.5929, -2.5929, -2.5929, -2.5929],
          [-2.5929, -2.5929, -2.5929, -2.5929, -2.5929, -2.5929],
          [-2.5929, -2.5929, -2.5929, -2.5929, -2.5929, -2.5929],
          [-2.5929, -2.5929, -2.5929, -2.5929, -2.5929, -2.5929]],

         [[-2.9530, -1.5942, -0.4920, -0.4920, -0.4920, -0.4920],
          [-0.4920, -0.4920, -0.4920, -0.4920, -0.4920, -0.4920],
          [-0.4920, -0.4920, -0.4920, -0.4920, -0.4920, -0.4920],
      